In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
import win32com.client as win32
import time  # Для измерения времени выполнения
import shutil
import re
from glob import glob
# from glob import glob
import gc
from datetime import timedelta

# Функция для форматирования времени в часы, минуты и секунды
def format_elapsed_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours)} часа(ов) {int(minutes)} минут(ы) {seconds:.2f} секунд"

# Функция для проверки, является ли файл скрытым (для Windows)
def is_hidden(file_path):
    try:
        # Получаем атрибуты файла
        file_attributes = os.stat(file_path).st_file_attributes
        # Проверяем, установлен ли флаг "скрытый"
        return file_attributes & 2 != 0  # 2 соответствует атрибуту "скрытый"
    except Exception:
        # Если возникла ошибка, считаем файл не скрытым
        return False

# Функция для форматирования даты в строковый формат 'YYYY-MM-DD'
def format_date_column(df, date_column):
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors='coerce').dt.strftime('%Y-%m-%d')
    return df

# Функция для подключения к SQL Server с аутентификацией Windows
def connect_to_sql(server, database):
    connection_string = (
        f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )
    engine = create_engine(connection_string)
    return engine

# Функция для обработки ошибок и замены их на null
def handle_errors(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(lambda x: None if isinstance(x, str) and x.strip() == '' else x)
    return df

In [2]:
FOLDER_PATH = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!")
FOLDER_PATH_FEATURES = r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям"
FOLDER_PATH_FOR_DB= os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям")
FOLDER_PATH_DUDL = os.path.normpath(r"\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ")

SQL_SERVER = "cl01sql"
SQL_DATABASE_DBREPORT = "DBReport"
SQL_DATABASE_DBPARTNERS = "DBPartners"

In [3]:
print("Начинаем собирать Базу Данных...")
start_all_time = time.time()
engine = connect_to_sql(SQL_SERVER, SQL_DATABASE_DBPARTNERS)

Начинаем собирать Базу Данных...


In [4]:
# 4. Получить данные из файла "Справочник.xlsx"
try:
    print("Начинаем получать данные для Справочника...")
    start_time = time.time()  # Запускаем таймер
    file_path_reference = os.path.join(FOLDER_PATH, "Справочник.xlsx")

    if os.path.exists(file_path_reference):
        # Список столбцов, которые нужно взять из файла
        columns_to_read = [
            "Артикул", "Артикул OZ", "Наименование", "Коллекция",
            "Бренд", "Размер", "Сезон", "Направление", "Розничный отдел",
            "Модель", "Группа", "Бизнес-группа", "Техсегмент",
            "Байер", "Две последние коллекции", "Основной артикул", "Ответственный за группу", "Себестоимость с НДС",
            "Процент выкупа", "НДС", "Группа для отчетов"
        ]

        # Типы данных для столбцов
        column_dtypes = {
            "Артикул": str,
            "Артикул OZ": str,
            "Наименование": str,
            "Коллекция": str,
            "Размер": str,
            "Бренд": str,
            "Сезон": str,
            "Направление": str,
            "Розничный отдел": str,
            "Модель": str,
            "Группа": str,
            "Бизнес-группа": str,
            "Техсегмент": str,
            "Байер": str,
            "Две последние коллекции": str,
            "Основной артикул": str,
            "Ответственный за группу": str,
            "Себестоимость с НДС": float,
            "Процент выкупа": float,
            "НДС": int,
            "Группа для отчетов": str
        }

        # Чтение файла с указанием нужных столбцов и типов данных
        df_reference = pd.read_excel(
            file_path_reference,
            sheet_name="Выгрузка для справочника",
            engine="openpyxl",
            usecols=columns_to_read,
            dtype=column_dtypes
        )

        # Удаление дубликатов
        df_reference = df_reference.drop_duplicates(subset=['Артикул', "Артикул OZ"])

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Справочник:")
        print(df_reference.head())

        # Сохраняем результат
        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Справочника успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Справочник.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Справочника: {e}")

Начинаем получать данные для Справочника...
Первые 5 строк таблицы Справочник:
    Артикул                         Наименование Размер Коллекция Бренд Сезон  \
0  W9009967  Полуботинки женские зимние ZL25AW-5     38    2025AW  kari  зима   
1  W9009967  Полуботинки женские зимние ZL25AW-5     36    2025AW  kari  зима   
2  W9009967  Полуботинки женские зимние ZL25AW-5     40    2025AW  kari  зима   
3  W9009967  Полуботинки женские зимние ZL25AW-5     37    2025AW  kari  зима   
4  W9009967  Полуботинки женские зимние ZL25AW-5     41    2025AW  kari  зима   

     Направление Розничный отдел    Модель Бизнес-группа  ... Техсегмент  \
0  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
1  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
2  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
3  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
4  Женская обувь   Женская обувь  ZL25AW-5         Обу

In [5]:
# 7. Создание таблицы "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу ВсегоРазмеров...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Артикул", "Размер"]
    for col in required_columns:
        if col not in df_reference.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_reference.")
            exit()

    # Очищаем столбец "Размер":
    # - Преобразуем в строковый формат
    # - Удаляем лишние пробелы
    # - Заменяем пустые строки на None
    df_reference["Размер"] = df_reference["Размер"].astype(str).str.strip().replace('', None)

    # Создаем DataFrame с количеством размеров для каждого артикула
    df_reference_unique = (
        df_reference
        .drop_duplicates(subset=["Артикул", "Размер"])  # Удаляем дубликаты Артикул-Размер
        .groupby("Артикул")["Размер"]  # Группируем по артикулу
        .apply(lambda sizes: len(sizes.dropna().unique()) if len(sizes.dropna()) > 0 else 1)  # Подсчитываем размеры
        .reset_index(name="Всего размеров")
    )

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВсегоРазмеров:")
    print(df_reference_unique.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВсегоРазмеров успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВсегоРазмеров: {e}")

Начинаем создавать таблицу ВсегоРазмеров...
Первые 5 строк таблицы ВсегоРазмеров:
    Артикул  Всего размеров
0  00001851               1
1  00001852               1
2  00001855               1
3  00001856               1
4  00001931               1
Таблица ВсегоРазмеров успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 37.39 секунд


In [6]:
# 6. Получить данные таблицы с SQL (РазмерыНаАгрегаторе)
try:
    print("Начинаем получать данные для РазмеровНаАгрегаторе...")
    start_time = time.time()  # Запускаем таймер
    query_sizes = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], COUNT(DISTINCT(a.[INVENTSIZEID])) AS [Колво размеров]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=2)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_sizes = pd.read_sql(query_sizes, engine)
    df_sizes = format_date_column(df_sizes, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы РазмерыНаАгрегаторе:")
    print(df_sizes.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для РазмеровНаАгрегаторе: {e}")

Начинаем получать данные для РазмеровНаАгрегаторе...
Первые 5 строк таблицы РазмерыНаАгрегаторе:
         Дата   Артикул  Колво размеров
0  2025-10-26  00006170               2
1  2025-10-26  00006410               2
2  2025-10-26  00128845               4
3  2025-10-26  00146325               6
4  2025-10-26  00146326               6
Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 28.76 секунд


In [7]:
# 8. Связать "РазмерыНаАгрегаторе" с "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу Дистрибуция...")
    start_time = time.time()  # Запускаем таймер

    # Объединяем таблицы по полю "Артикул"
    df_distribution = pd.merge(df_sizes, df_reference_unique, on="Артикул", how="left")

    # Вычисляем дистрибуцию с проверкой на деление на ноль
    df_distribution["Дистрибуция"] = df_distribution.apply(
        lambda row: row["Колво размеров"] / row["Всего размеров"] if row["Всего размеров"] != 0 else 0,
        axis=1
    )

    # Оставляем только нужные столбцы
    df_distribution = df_distribution[["Дата", "Артикул", "Дистрибуция"]]

    # Форматирование даты
    df_distribution = format_date_column(df_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Дистрибуция:")
    print(df_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Дистрибуция успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Дистрибуция: {e}")

Начинаем создавать таблицу Дистрибуция...
Первые 5 строк таблицы Дистрибуция:
         Дата   Артикул  Дистрибуция
0  2025-10-26  00006170     0.333333
1  2025-10-26  00006410     0.333333
2  2025-10-26  00128845     1.000000
3  2025-10-26  00146325     1.000000
4  2025-10-26  00146326     1.000000
Таблица Дистрибуция успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 24.33 секунд


In [8]:
# 2. Получить данные таблицы с SQL (Остатки)
try:
    print("Начинаем получать данные для Остатков...")
    start_time = time.time()  # Запускаем таймер
    query_stock = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], SUM(a.[free_to_sell_amount]) AS [Остаток Агрегатора]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=2)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_stock = pd.read_sql(query_stock, engine)
    df_stock = format_date_column(df_stock, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатков:")
    print(df_stock.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для Остатков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для Остатков: {e}")

Начинаем получать данные для Остатков...
Первые 5 строк таблицы Остатков:
         Дата   Артикул  Остаток Агрегатора
0  2025-10-26  00006170                   2
1  2025-10-26  00006410                   0
2  2025-10-26  00128845                 175
3  2025-10-26  00146325                 191
4  2025-10-26  00146326                 180
Данные для Остатков успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 28.48 секунд


In [9]:
# 9. Связать "Остатки" с "Дистрибуция"
try:
    print("Начинаем создавать таблицу Остатки с дистрибуцией...")
    start_time = time.time()  # Запускаем таймер
    df_stock_with_distribution = pd.merge(df_stock, df_distribution, on=["Дата", "Артикул"], how="left")
    df_stock_with_distribution = format_date_column(df_stock_with_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатки с дистрибуцией:")
    print(df_stock_with_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Остатки с дистрибуцией успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Остатки с дистрибуцией: {e}")

Начинаем создавать таблицу Остатки с дистрибуцией...
Первые 5 строк таблицы Остатки с дистрибуцией:
         Дата   Артикул  Остаток Агрегатора  Дистрибуция
0  2025-10-26  00006170                   2     0.333333
1  2025-10-26  00006410                   0     0.333333
2  2025-10-26  00128845                 175     1.000000
3  2025-10-26  00146325                 191     1.000000
4  2025-10-26  00146326                 180     1.000000
Таблица Остатки с дистрибуцией успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 3.18 секунд


In [10]:
del df_stock

In [ ]:
import pandas as pd
import glob
import os
import datetime
import pyodbc

# Папки
path_voronka = r"Z:\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!\ВЫГРУЗКА воронка Озон"
path_zatraty = r"Z:\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!\Затраты\Озон. Затраты из Аналитики"

# === 1. ВОРОНКА ===
df_voronka_list = []
files_voronka = glob.glob(os.path.join(path_voronka, "analytics_report_*.xlsx"))

for file in files_voronka:
    # достаём дату из имени файла
    fname = os.path.basename(file)
    try:
        report_date = datetime.datetime.strptime(fname.split("_")[2], "%Y-%m-%d").date() - datetime.timedelta(days=1)
    except Exception:
        continue

    df = pd.read_excel(file, engine='calamine')

    # Чистим "Позиция в поиске и каталоге" от запятых
    df["Позиция в поиске и каталоге"] = (
        df["Позиция в поиске и каталоге"]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )

    # df["Позиция в поиске и каталоге"] = pd.to_numeric(df["Позиция в поиске и каталоге"], errors="coerce")
    # print(df.head(20))
    # типы
    df = df.astype({
        "Артикул": "string",
        "Показы, всего": "Int64",
        "Показы на карточке товара": "Int64",
        "Показы в поиске и каталоге": "Int64",
        "Позиция в поиске и каталоге": "float64",
        "В корзину, всего": "Int64",
        "Заказано товаров": "Int64",
        "Отменено товаров": "Int64",
        "Доставлено товаров": "Int64",
        "Возвращено товаров": "Int64",
        "Заказано на сумму": "float64",
        "В корзину из карточки товара": "Int64"
    })

    df["Дата"] = report_date
    if report_date == '01.12.2025':
        print(df["Показы, всего"].sum())

    df["Выкупили ШТ"] = df["Заказано товаров"] - df["Отменено товаров"] - df["Возвращено товаров"]
    df["Артикул"] = df["Артикул"].astype(str).str.split("-").str[0]

    df_voronka_list.append(df)
df_voronka = pd.concat(df_voronka_list, ignore_index=True)

sum_cols_all = [
    "Показы, всего",
    "Показы на карточке товара",
    "Показы в поиске и каталоге",
    "Позиция в поиске и каталоге",          # суммируем, как в вашем примере
    "В корзину, всего",
    "Заказано товаров",
    "Отменено товаров",
    "Доставлено товаров",
    "Возвращено товаров",
    "Заказано на сумму",
    "В корзину из карточки товара",
    "В корзину из поиска или каталога",     # если есть в выгрузке
    "Выкупили ШТ"
]
sum_cols = [c for c in sum_cols_all if c in df_voronka.columns]
temp_df.rename(columns={
                        "Артикул": "Артикул",
                        "Продажи, ₽": "Рекламные заказано на сумму",
                        "Показы": "Рекламные показы",
                        "Клики": "Рекламные показы на карточке товара",
                        "Заказы, шт": "Рекламные заказано товаров"
                    }, inplace=True, errors="ignore")
# 2) безопасно приводим эти метрики к числам (NaN -> 0 перед суммированием)
for c in sum_cols:
    df_voronka[c] = pd.to_numeric(df_voronka[c], errors="coerce").fillna(0)

# 3) нечисловые поля, которые логично брать первыми в группе
first_cols_all = ["Тип товара", "Товары", "Модель", "Ozon ID"]
first_cols = [c for c in first_cols_all if c in df_voronka.columns]

# 4) готовим словарь агрегаций
agg_dict = {c: "sum" for c in sum_cols}
agg_dict.update({c: "first" for c in first_cols})

# 5) группируем и получаем одну строку на (Дата, Артикул)
df_voronka = (
    df_voronka
    .groupby(["Дата", "Артикул"], as_index=False)
    .agg(agg_dict)
)

In [18]:
df_voronka[df_voronka['Дата'] == datetime.date(2025, 12, 1)]['Показы, всего'].sum()

np.int64(132002253)

In [19]:
df_voronka['Дата'].unique()

array([datetime.date(2025, 11, 9), datetime.date(2025, 11, 10),
       datetime.date(2025, 11, 11), datetime.date(2025, 11, 12),
       datetime.date(2025, 11, 13), datetime.date(2025, 11, 14),
       datetime.date(2025, 11, 15), datetime.date(2025, 11, 16),
       datetime.date(2025, 11, 17), datetime.date(2025, 11, 18),
       datetime.date(2025, 11, 19), datetime.date(2025, 11, 20),
       datetime.date(2025, 11, 21), datetime.date(2025, 11, 22),
       datetime.date(2025, 11, 23), datetime.date(2025, 11, 24),
       datetime.date(2025, 11, 25), datetime.date(2025, 11, 26),
       datetime.date(2025, 11, 27), datetime.date(2025, 11, 28),
       datetime.date(2025, 11, 29), datetime.date(2025, 11, 30),
       datetime.date(2025, 12, 1), datetime.date(2025, 12, 2),
       datetime.date(2025, 12, 3), datetime.date(2025, 12, 4),
       datetime.date(2025, 12, 5), datetime.date(2025, 12, 6),
       datetime.date(2025, 12, 7), datetime.date(2025, 12, 8),
       datetime.date(2025, 12, 9),

In [26]:
df_voronka

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,В корзину из поиска или каталога,Выкупили ШТ,Тип товара,Товары,Модель,Ozon ID
0,2025-11-09,00006000,2,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки T.TACCARDI,Балетки T.TACCARDI,149391001
1,2025-11-09,00006020,2,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки T.TACCARDI,Балетки T.TACCARDI,149393765
2,2025-11-09,00006080,1,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки T.TACCARDI,Балетки T.TACCARDI,149390933
3,2025-11-09,000060H0,1,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки Pierre Cardin,Балетки Pierre Cardin,149354563
4,2025-11-09,00006130,1,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки T.TACCARDI,Балетки T.TACCARDI,149393825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332060,2025-12-25,y9808110,115,3,25,1121.08,0,0,0,0,0,0.0,0,0,0,Шорты,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1997933024
2332061,2025-12-25,y9808120,50,1,14,737.05,0,0,0,0,0,0.0,0,0,0,Шорты,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1997932906
2332062,2025-12-25,y9808130,21,0,8,1222.40,0,0,0,0,0,0.0,0,0,0,Шорты,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1934644792
2332063,2025-12-25,y9808140,23,0,6,404.40,0,0,0,0,0,0.0,0,0,0,Шорты,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1950208470


In [27]:
df_voronka[df_voronka.duplicated(subset=["Артикул", "Дата", "Ozon ID"], keep=False)].sort_values(by='Артикул')

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,В корзину из поиска или каталога,Выкупили ШТ,Тип товара,Товары,Модель,Ozon ID


In [28]:
temp = df_voronka.drop_duplicates(subset=["Артикул", "Дата", "Ozon ID"])
temp[temp['Дата'] == datetime.date(2025, 12, 1)]['Показы, всего'].sum()

np.int64(132002253)

In [29]:
# === 2. ЗАТРАТЫ ===
df_zatraty_list = []
files_zatraty = glob.glob(os.path.join(path_zatraty, "*.csv"))

for f in files_zatraty:
    # дата из имени файла
    fname = os.path.basename(f).replace(".csv", "")
    file_date = pd.to_datetime(fname, dayfirst=True, errors="coerce")

    # читаем csv
    df = pd.read_csv(f, sep=";", skiprows=2)

    # чистим названия колонок
    df.columns = [c.replace(".csv","") if ".csv" in c else c for c in df.columns]

    # вставляем дату из имени файла
    df["Дата"] = file_date

    # переименования
    df = df.rename(columns={
        "Тип продвижения": "ТипАктивности",
        "Расход, ₽, с НДС": "Расход, ₽"
    })

    df_zatraty_list.append(df)

df_zatraty = pd.concat(df_zatraty_list, ignore_index=True)

In [30]:
# === 3. SQL ЦЕНЫ ===
sql = f"""
select 'OZ' as AGREGATOR, DT, ITEMID, PRICE
from [DBPartners].[dbo].[WblmRepPriceDiscountOzReport]
where dt >= '{(pd.Timestamp.today() - pd.DateOffset(months=2)).strftime("%Y-%m-%d")}'
"""
df_prices = pd.read_sql(sql, engine)

df_prices = df_prices.groupby(["DT", "ITEMID"], as_index=False).agg({"PRICE": "max"})
df_prices = df_prices.rename(columns={"DT": "Дата", "ITEMID": "Артикул", "PRICE": "Цена"})

In [31]:
df_reference = df_reference.drop_duplicates(subset=["Артикул"])
df_reference = df_reference[["Артикул", "Бизнес-группа", "Направление", "Розничный отдел", "Группа", "Модель", "Бренд", "Коллекция", "Сезон", "Себестоимость с НДС", "Процент выкупа", "Две последние коллекции", 'Артикул OZ', 'Наименование', 'Техсегмент', 'Байер', 'Основной артикул', 'НДС', 'Ответственный за группу', 'Группа для отчетов']]
df_reference

,Артикул,Бизнес-группа,Направление,Розничный отдел,Группа,Модель,Бренд,Коллекция,Сезон,Себестоимость с НДС,Процент выкупа,Две последние коллекции,Артикул OZ,Наименование,Техсегмент,Байер,Основной артикул,НДС,Ответственный за группу,Группа для отчетов
0,W9009967,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,ZL25AW-5,kari,2025AW,зима,1339.2297,0.860759,2025AW,2445277397,Полуботинки женские зимние ZL25AW-5,flat (L),Коновалова А.,W9009967,20,Гусева Дарья,Обувь
6,W9009647,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,MYZ25AW-124,kari,2025AW,зима,1242.9769,0.875000,2025AW,2608453582,Полуботинки женские зимние MYZ25AW-124,flat (L),Коновалова А.,W9009647,20,Гусева Дарья,Обувь
11,W9009968,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,ZL25AW-5A,kari,2025AW,зима,1339.0063,0.821429,2025AW,2445277632,Полуботинки женские зимние ZL25AW-5A,flat (L),Коновалова А.,W9009968,20,Гусева Дарья,Обувь
17,W9059533,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,K1522LW-9J,kari,2025AW,зима,829.7751,0.892308,2025AW,2445275141,Полуботинки женские зимние K1522LW-9J,flat (AM),Коновалова А.,W9059533,20,Гусева Дарья,Обувь
23,W9059534,Обувь,Женская обувь,Женская обувь,001 Полуботинки женские зимние,K1522LW-9JA,kari,2025AW,зима,828.9668,0.600000,2025AW,2445275943,Полуботинки женские зимние K1522LW-9JA,flat (AM),Коновалова А.,W9059534,20,Гусева Дарья,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
702876,ya102020,Принадлежности для спорта и активного отдыха,Летний спорт,Велосипеды,ya1 Аксессуары для велосипедов,K7379,KariKids,2023SS,лето,55.1370,0.934197,"2022SS,2023SS",NaN,Корзина для велосипеда черно-белая K7379,NaN,Бордачук Е. Спорт,ya102020,20,Шляпин Алексей,NaN
702877,ya106020,Принадлежности для спорта и активного отдыха,Летний спорт,Велосипеды,ya1 Аксессуары для велосипедов,K10997,Kari KIDS,2024SS,лето,78.5352,0.934197,2024SS,NaN,Корзина для велосипеда черная K10997,NaN,Бордачук Е. Спорт,ya106020,20,Шляпин Алексей,NaN
702878,ya108000,Принадлежности для спорта и активного отдыха,Летний спорт,Велосипеды,ya1 Аксессуары для велосипедов,JKPB254,kari,2025SS,лето,155.2165,0.934197,2025SS,NaN,"Ручка-толкатель для велосипеда 12""-16"" JKPB254",NaN,Бордачук Е. Спорт,ya108000,20,Шляпин Алексей,NaN
702879,ya108010,Принадлежности для спорта и активного отдыха,Летний спорт,Велосипеды,ya1 Аксессуары для велосипедов,K13760,kari,2025SS,лето,95.6673,0.934197,2025SS,NaN,Корзина для велосипеда голубая K13760,NaN,Бордачук Е. Спорт,ya108010,20,Шляпин Алексей,NaN


In [32]:
df_zatraty['SKU']

0          1066351626
1          1869030963
2          1427091062
3          1056252755
4          1097494305
              ...    
2559623    1659022062
2559624    1871101118
2559625    1428832580
2559626    1645388852
2559627    2430315599
Name: SKU, Length: 2559628, dtype: int64

In [33]:
df_voronka

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,В корзину из поиска или каталога,Выкупили ШТ,Тип товара,Товары,Модель,Ozon ID
0,2025-11-09,00006000,2,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки T.TACCARDI,Балетки T.TACCARDI,149391001
1,2025-11-09,00006020,2,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки T.TACCARDI,Балетки T.TACCARDI,149393765
2,2025-11-09,00006080,1,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки T.TACCARDI,Балетки T.TACCARDI,149390933
3,2025-11-09,000060H0,1,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки Pierre Cardin,Балетки Pierre Cardin,149354563
4,2025-11-09,00006130,1,0,0,0.00,0,0,0,0,0,0.0,0,0,0,Балетки,Балетки T.TACCARDI,Балетки T.TACCARDI,149393825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332060,2025-12-25,y9808110,115,3,25,1121.08,0,0,0,0,0,0.0,0,0,0,Шорты,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1997933024
2332061,2025-12-25,y9808120,50,1,14,737.05,0,0,0,0,0,0.0,0,0,0,Шорты,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1997932906
2332062,2025-12-25,y9808130,21,0,8,1222.40,0,0,0,0,0,0.0,0,0,0,Шорты,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1934644792
2332063,2025-12-25,y9808140,23,0,6,404.40,0,0,0,0,0,0.0,0,0,0,Шорты,Шорты Kari мужские спортивные,Шорты Kari мужские спортивные,1950208470


In [35]:
df

,SKU,ТипАктивности,ID кампании,"Расход, ₽","ДРР, %","Продажи, ₽","Заказы, шт","CTR, %",Показы,Клики,В корзину,"Конверсия в корзину, %","Затраты на заказ, ₽","Стоимость клика, ₽",Дата
0,1882483239,Оплата за клик,18354406,"0,00",NaN,"0,00",0,"0,00","10,00","0,00","0,00",NaN,NaN,NaN,2025-10-31
1,1076245943,Оплата за клик,18409032,"0,00",NaN,"0,00",0,"0,00","24,00","0,00","0,00",NaN,NaN,NaN,2025-10-31
2,1391270634,Оплата за клик,18352401,"0,00",NaN,"0,00",0,"0,00","16,00","0,00","0,00",NaN,NaN,NaN,2025-10-31
3,543799580,Оплата за клик,18354402,"0,00",NaN,"0,00",0,"0,00","94,00","0,00","0,00",NaN,NaN,NaN,2025-10-31
4,2545012465,Оплата за клик,18351574,"59,16",NaN,"0,00",0,"4,00","50,00","2,00","0,00","0,00",NaN,"29,58",2025-10-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42440,1659022062,Оплата за клик,18291098,"0,00","0,00","0,00",0,"0,00","0,00","0,00","0,00","0,00","0,00","0,00",2025-10-31
42441,1871101118,Оплата за клик,17423369,"0,00","0,00","0,00",0,"0,00","0,00","0,00","0,00","0,00","0,00","0,00",2025-10-31
42442,1428832580,Оплата за клик,18289352,"0,00","0,00","0,00",0,"0,00","0,00","0,00","0,00","0,00","0,00","0,00",2025-10-31
42443,1645388852,Оплата за клик,18030911,"0,00","0,00","0,00",0,"0,00","0,00","0,00","0,00","0,00","0,00","0,00",2025-10-31


In [97]:
df_zatraty

,SKU,ТипАктивности,ID кампании,"Расход, ₽","ДРР, %","Продажи, ₽","Заказы, шт","CTR, %",Показы,Клики,"Стоимость заказа, ₽","Стоимость клика, ₽",Корзины,"Конверсия в корзину, %",Дата,В корзину,"Затраты на заказ, ₽"
0,1066351626,Трафареты,16567480,"6,15",NaN,"0,00",0,"3,77",53,2,NaN,"3,07",0.0,"0,00",2025-08-01,NaN,NaN
1,1869030963,Трафареты,16588991,"0,00",NaN,"0,00",0,"0,00",82,0,NaN,NaN,0.0,NaN,2025-08-01,NaN,NaN
2,1427091062,Трафареты,16589274,"1,07",NaN,"0,00",0,"0,78",129,1,NaN,"1,07",0.0,"0,00",2025-08-01,NaN,NaN
3,1056252755,Трафареты,16490561,"665,45",NaN,"0,00",0,"2,37",1899,45,NaN,"14,79",6.0,"13,33",2025-08-01,NaN,NaN
4,1097494305,Трафареты,16588951,"0,00",NaN,"0,00",0,"0,00",40,0,NaN,NaN,0.0,NaN,2025-08-01,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2559623,1659022062,Оплата за клик,18291098,"0,00","0,00","0,00",0,"0,00","0,00","0,00",NaN,"0,00",NaN,"0,00",2025-10-31,"0,00","0,00"
2559624,1871101118,Оплата за клик,17423369,"0,00","0,00","0,00",0,"0,00","0,00","0,00",NaN,"0,00",NaN,"0,00",2025-10-31,"0,00","0,00"
2559625,1428832580,Оплата за клик,18289352,"0,00","0,00","0,00",0,"0,00","0,00","0,00",NaN,"0,00",NaN,"0,00",2025-10-31,"0,00","0,00"
2559626,1645388852,Оплата за клик,18030911,"0,00","0,00","0,00",0,"0,00","0,00","0,00",NaN,"0,00",NaN,"0,00",2025-10-31,"0,00","0,00"


In [427]:
import pandas as pd
import numpy as np

# ===== Нормализация и парсинг =====
NBSP = '\u00A0'

def norm_date(s: pd.Series) -> pd.Series:
    d = pd.to_datetime(s, errors='coerce')
    try:
        d = d.dt.tz_localize(None)
    except Exception:
        pass
    return d.dt.normalize()

def norm_code(s: pd.Series) -> pd.Series:
    return (s.astype(str)
              .str.replace(NBSP, '', regex=False)
              .str.strip()
              .str.upper())

def norm_code_alnum(s: pd.Series) -> pd.Series:
    # только A–Z и 0–9 — удобно для OZ/SKU
    return norm_code(s).str.replace(r'[^0-9A-Z]+', '', regex=True)

def to_money(s: pd.Series) -> pd.Series:
    # «1 234,56 ₽», «1.234,56», «1234,56» → 1234.56
    ss = (s.astype(str)
            .str.replace(NBSP, '', regex=False)
            .str.replace(' ',  '', regex=False)
            .str.replace('₽',  '', regex=False))
    ss = ss.str.replace(r'(?<=\d)\.(?=\d{3}(?:\D|$))', '', regex=True)  # 1.234,56 → 1234,56
    ss = ss.str.replace(',', '.', regex=False)
    ss = ss.str.replace(r'[^0-9\.\-\+eE]', '', regex=True)
    return pd.to_numeric(ss, errors='coerce')

# ===== Основная функция =====
def merge_voronka_costs_preserve_impressions(
    df_voronka: pd.DataFrame,
    df_costs: pd.DataFrame,
    df_prices: pd.DataFrame | None = None,
    *,
    left_key_candidates=('Ozon ID','OZON ID','OZON_ID','Артикул OZ','Артикул'),
    right_key='SKU',
    spend_candidates=('Расход, ₽','Расход, руб','Расход, Р','Расход'),
    preserve_cols=('Показы, всего',),    # инвариант по этим левым метрикам
    add_tail=True,                        # добавлять ли «хвост» неприсоединившихся затрат
    type_col_candidates=('ТипАктивности','Тип активности','Раздел'),
    join_types_sep=' и '
) -> pd.DataFrame:
    """Склейка «воронки» с затратами (и прайсом, опционально) без флагов,
    с единственной колонкой 'ТипАктивности' из затрат и сохранением инвариантов."""

    # --- 0) Копии + нормализация дат ---
    df_v = df_voronka.copy()
    df_z = df_costs.copy()
    df_p = None if df_prices is None else df_prices.copy()

    for dframe in (df_v, df_z) + ((df_p,) if df_p is not None else ()):
        if dframe is not None and 'Дата' in dframe.columns:
            dframe['Дата'] = norm_date(dframe['Дата'])

    # --- 1) Инварианты по левым метрикам (например, «Показы, всего») ---
    def _num(s): return pd.to_numeric(s, errors='coerce')
    baseline = {c: (_num(df_v[c]).sum() if c in df_v.columns else None) for c in preserve_cols}

    # --- 2) Выбор левого ключа OZ/SKU ---
    left_key = next((c for c in left_key_candidates if c in df_v.columns), None)
    if left_key is None:
        raise KeyError(f"Во воронке нет ни одного ключа из {left_key_candidates}")

    # --- 3) Подготовка затрат (правая таблица) ---
    if right_key not in df_z.columns:
        raise KeyError(f"В затратах нет колонки {right_key}")
    spend_col = next((c for c in spend_candidates if c in df_z.columns), None)
    if spend_col is None:
        raise KeyError(f"В затратах нет денежной колонки из {spend_candidates}")

    # 3.1 Ключи и деньги
    df_z[right_key] = norm_code(df_z[right_key])
    df_z['__KEY__'] = norm_code_alnum(df_z[right_key])
    df_z[spend_col] = to_money(df_z[spend_col]).astype('float64')

    # 3.2 Нормализация типа активности в затратах (если он есть)
    z_type_col = next((c for c in type_col_candidates if c in df_z.columns), None)
    if z_type_col is not None:
        norm_map = {
            'ТОП': 'Вывод в топ',
            'Единая Ставка': 'Автоматическое',
            'Единая ставка': 'Автоматическое',
            'Ручная Ставка': 'Аукцион',
            'Ручная ставка': 'Аукцион',
            'АУКЦИОН': 'Аукцион',
            'АВТОМАТИЧЕСКОЕ': 'Автоматическое',
        }
        df_z[z_type_col] = (df_z[z_type_col].astype(str).str.strip()
                            .replace(norm_map)).replace({'': np.nan})

    # 3.3 Метрики из затрат (добавочные)
    extra_metrics = [c for c in ['Показы','Клики','Заказы, шт','Продажи, ₽'] if c in df_z.columns]
    z_metrics = [spend_col] + extra_metrics

    # 3.4 Суммы по (Дата, __KEY__)
    z_agg = (df_z.groupby(['Дата','__KEY__'], as_index=False)[z_metrics].sum(min_count=1))
    total_costs = z_agg[spend_col].sum()

    # 3.5 ЕДИНАЯ колонка 'ТипАктивности' по (Дата, __KEY__) (без флагов и без суффиксов)
    if z_type_col is not None:
        typeset = (df_z[['Дата','__KEY__', z_type_col]]
                   .dropna(subset=[z_type_col])
                   .groupby(['Дата','__KEY__'])[z_type_col]
                   .apply(lambda s: sorted(pd.unique(s.dropna().tolist())))
                   .reset_index(name='__typeset__'))

        def _join_types(v):
            if not v: return np.nan
            if len(v) == 1: return v[0]
            return join_types_sep.join(v)

        types_info = typeset.copy()
        types_info['ТипАктивности'] = types_info['__typeset__'].apply(_join_types)
        types_info = types_info.drop(columns=['__typeset__'])

        # чтобы избежать коллизии имён при merge, если слева есть 'ТипАктивности',
        # временно назовём правую колонку иначе
        right_type_col = 'ТипАктивности_from_costs'
        types_info = types_info.rename(columns={'ТипАктивности': right_type_col})

        # присоединяем типовую инфу к агрегату затрат
        z_agg_full = z_agg.merge(types_info, on=['Дата','__KEY__'], how='left')
    else:
        z_agg_full = z_agg.copy()
        right_type_col = None

    # --- 4) Левый нормализованный ключ и маркировка исходных строк ---
    df_v['__KEY__'] = norm_code_alnum(norm_code(df_v[left_key]))
    df_v['IS_TAIL'] = 0
    df_v['SRC_ROW_ID'] = np.arange(len(df_v), dtype='int64')

    # --- 5) Основной LEFT m:1 join (без m×n) ---
    df = df_v.merge(z_agg_full, on=['Дата','__KEY__'], how='left', validate='m:1')

    # --- 6) Сведение единой 'ТипАктивности' без суффиксов ---
    # если слева уже есть 'ТипАктивности', заполним в нём NaN значениями из затрат
    if right_type_col is not None:
        if 'ТипАктивности' in df.columns:
            # заполняем только пропуски
            df['ТипАктивности'] = df['ТипАктивности'].where(df['ТипАктивности'].notna(), df[right_type_col])
            df.drop(columns=[right_type_col], inplace=True)
        else:
            # переименуем правую колонку в целевое имя
            df.rename(columns={right_type_col: 'ТипАктивности'}, inplace=True)

    # --- 7) Контроль инвариантов по левым метрикам (только исходные строки) ---
    for c in preserve_cols:
        if c in df.columns and baseline[c] is not None:
            after = _num(df.loc[df['IS_TAIL']==0, c]).sum()
            msg = "[OK]" if np.isclose(after, baseline[c], rtol=1e-9, atol=1e-6) else "[WARN]"
            print(f"{msg} Инвариант '{c}': до={baseline[c]:,.2f} | после={after:,.2f}")

    # --- 8) Добавляем «хвост» затрат, у которых нет пары на ту же дату ---
    tail_rows = 0
    if add_tail:
        left_keys = df[['Дата','__KEY__']].drop_duplicates()
        right_cols_only = [c for c in z_agg_full.columns if c not in ('Дата','__KEY__')]
        miss = (z_agg_full.merge(left_keys, on=['Дата','__KEY__'], how='left', indicator=True)
                          .loc[lambda x: x['_merge']=='left_only', ['Дата','__KEY__'] + right_cols_only])
        if not miss.empty:
            tail = miss.copy()
            # если есть временная колонка типа из затрат — сразу переименуем в целевую
            if right_type_col and right_type_col in tail.columns:
                tail.rename(columns={right_type_col: 'ТипАктивности'}, inplace=True)
            # каркас из df: заполняем недостающие левые колонки NaN
            for col in df.columns:
                if col not in tail.columns:
                    tail[col] = np.nan
            tail['IS_TAIL'] = 1
            # чтобы видно было, какой ключ пришёл справа
            tail[left_key] = tail['__KEY__']
            # выравниваем порядок
            tail = tail[df.columns]
            df = pd.concat([df, tail], ignore_index=True)
            tail_rows = len(tail)

    # --- 9) Подтягиваем прайс (опционально) ---
    if df_p is not None and {'Дата','Артикул'}.issubset(df.columns) and {'Дата','Артикул'}.issubset(df_p.columns):
        df_p['Артикул'] = norm_code(df_p['Артикул'])
        df = df.merge(df_p, on=['Дата','Артикул'], how='left', validate='m:1')

    # --- 10) Финальные сверки (расходы и сохранённые метрики) ---
    final_costs = pd.to_numeric(df[spend_col], errors='coerce').fillna(0).sum()
    delta = final_costs - total_costs
    print(f"[CHECK] Расходы: в файле = {total_costs:,.2f} ₽ | в результате = {final_costs:,.2f} ₽ | Δ = {delta:,.2f} ₽")
    if tail_rows:
        print(f"[INFO] Добавлен хвост (IS_TAIL=1): {tail_rows} строк")

    for c in preserve_cols:
        if c in df.columns and baseline[c] is not None:
            after_src = _num(df.loc[df['IS_TAIL']==0, c]).sum()
            print(f"[CHECK] '{c}' (исходные строки): до={baseline[c]:,.2f} | после={after_src:,.2f}")

    # --- 11) Уборка служебных колонок ---
    df.drop(columns=['__KEY__'], inplace=True, errors='ignore')
    df.drop(columns=['IS_TAIL'], inplace=True, errors='ignore')
    df.drop(columns=['SRC_ROW_ID'], inplace=True, errors='ignore')

    df.rename(columns={
                        "Артикул": "Артикул",
                        "Продажи, ₽": "Рекламные заказано на сумму",
                        "Показы": "Рекламные показы",
                        "Клики": "Рекламные показы на карточке товара",
                        "Заказы, шт": "Рекламные заказано товаров"
                    }, inplace=True, errors="ignore")

    return df

df_funnel = merge_voronka_costs_preserve_impressions(
    df_voronka=df_voronka,
    df_costs=df_zatraty,
    df_prices=df_prices,                              # если надо — подайте прайс
    preserve_cols=('Показы, всего',),            # можно добавить и другие левые метрики
    add_tail=True
)


C:\Users\i.taldykin\AppData\Local\Temp\ipykernel_8256\2744943371.py:111: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  .apply(lambda s: sorted(pd.unique(s.dropna().tolist())))


[OK] Инвариант 'Показы, всего': до=5,380,483,905.00 | после=5,380,483,905.00


C:\Users\i.taldykin\AppData\Local\Temp\ipykernel_8256\2744943371.py:181: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, tail], ignore_index=True)


[CHECK] Расходы: в файле = 689,172,958.63 ₽ | в результате = 689,172,958.63 ₽ | Δ = 0.00 ₽
[INFO] Добавлен хвост (IS_TAIL=1): 2032762 строк
[CHECK] 'Показы, всего' (исходные строки): до=5,380,483,905.00 | после=5,380,483,905.00


In [428]:
df_funnel.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'В корзину из поиска или каталога',
       'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Ozon ID', 'Расход, ₽',
       'Рекламные показы', 'Рекламные показы на карточке товара',
       'Рекламные заказано товаров', 'Рекламные заказано на сумму',
       'ТипАктивности', 'Цена'],
      dtype='object')

In [429]:
df_funnel[df_funnel['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(8307669.5)

In [430]:
df_funnel[df_funnel['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.int64(132002253)

In [431]:
funnel_columns = df_funnel.columns

In [432]:
# Путь до папки
folder_path_weeks = os.path.join(FOLDER_PATH, "Затраты", "Озон. Затраты из Аналитики New Format")

# Собираем все .xlsx файлы
files = glob.glob(os.path.join(folder_path_weeks, "*.xlsx"))

df_list = []
df_list_union = []
for file in files:
    # --- достаём дату из названия файла ---
    filename = os.path.basename(file)  # например: "Аналитика продвижения_16.09.2025.xlsx"
    date_str = filename.split("_")[-1].replace(".xlsx", "")  # "16.09.2025"
    date_parsed = (pd.to_datetime(date_str, format="%d.%m.%Y") - timedelta(days=1)).strftime("%Y-%m-%d")

    # читаем, пропуская первую строку
    df_tmp = pd.read_excel(file, engine='calamine', skiprows=1)
    df_tmp_union = pd.read_excel(file, sheet_name='Union', engine='calamine', skiprows=1)

    # оставляем только нужные колонки
    cols_keep = ["SKU", "ID кампании", "Инструмент", "Место размещения"]
    cols_keep_union = ["SKU в продвижении", "SKU из объединенной карточки", "Продажи, ₽", "Заказы, шт"]
    df_tmp = df_tmp[cols_keep]
    df_tmp_union = df_tmp_union[cols_keep_union]

    # добавляем колонку "Дата"
    df_tmp["Дата"] = date_parsed
    df_tmp_union["Дата"] = date_parsed

    df_list.append(df_tmp)
    df_list_union.append(df_tmp_union)

# объединяем все файлы
df_all = pd.concat(df_list, ignore_index=True)
df_all_union = pd.concat(df_list_union, ignore_index=True)
df_all.rename(columns={'SKU': "Артикул OZ"},inplace=True)
df_all_union.rename(columns={'SKU в продвижении': "Артикул OZ"},inplace=True)
df_all["Артикул OZ"] = df_all["Артикул OZ"].astype(str)
df_all_union["Артикул OZ"] = df_all_union["Артикул OZ"].astype(str)

In [435]:
df_union_reference = pd.merge(df_reference[['Артикул', 'Артикул OZ']], df_all_union, on="Артикул OZ", how='right')
# df_union_reference[df_union_reference['Артикул'].isna()]
df_union_reference

,Артикул,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт",Дата
0,Y0806110,1607739332,823892917,301.0,1,2025-09-30
1,NaN,2438516299,2310030768,2508.0,1,2025-09-30
2,W0257950,1627853566,1627855021,755.0,1,2025-09-30
3,NaN,1627854626,1627852674,722.0,1,2025-09-30
4,NaN,1627857305,1829020570,2084.0,1,2025-09-30
...,...,...,...,...,...,...
953929,NaN,1267725462,351952384,5423.0,1,2025-10-30
953930,W8579567,2659157998,2659158129,3068.0,1,2025-10-30
953931,NaN,2810541646,2810540473,496.0,1,2025-10-30
953932,NaN,1830569193,1830568337,1908.0,1,2025-10-30


In [436]:
df_union_reference['Продажи, ₽'] = pd.to_numeric(df_union_reference['Продажи, ₽'], errors='coerce')
df_union_reference['Заказы, шт'] = pd.to_numeric(df_union_reference['Заказы, шт'], errors='coerce')

df_union_agg = (
    df_union_reference
    .groupby(['Артикул', 'Дата'], as_index=False)
    .agg({
        'Артикул OZ': 'first',   # любой один из группы
        'SKU из объединенной карточки': 'first',   # любой один из группы
        'Продажи, ₽': 'sum',     # суммируем деньги
        'Заказы, шт': 'sum',     # суммируем заказы
    })
)
df_union_agg

,Артикул,Дата,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт"
0,00206110,2025-10-02,149484604,149484603,1706.0,2
1,00206110,2025-10-03,149484604,149484602,853.0,1
2,00206110,2025-10-05,149484604,149484603,1706.0,2
3,00206110,2025-10-06,149484604,149484603,853.0,1
4,00206110,2025-10-07,149484604,149484450,1363.0,1
...,...,...,...,...,...,...
131165,Y9708130,2025-11-05,1950209178,1979141782,1191.0,1
131166,Y9708130,2025-11-15,1950209178,1950208719,2081.0,2
131167,Y9708130,2025-11-20,1950209178,1950208649,1536.0,1
131168,Y9708130,2025-12-13,1950209178,1950208649,1536.0,1


In [437]:
df_union_reference.columns

Index(['Артикул', 'Артикул OZ', 'SKU из объединенной карточки', 'Продажи, ₽',
       'Заказы, шт', 'Дата'],
      dtype='object')

In [438]:
df_union_reference.drop_duplicates(subset=['Артикул','Дата'])

,Артикул,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт",Дата
0,Y0806110,1607739332,823892917,301.0,1,2025-09-30
1,NaN,2438516299,2310030768,2508.0,1,2025-09-30
2,W0257950,1627853566,1627855021,755.0,1,2025-09-30
17,U4801270,794548253,794561094,2456.0,2,2025-09-30
22,36507140,1639853574,1608236975,3777.0,1,2025-09-30
...,...,...,...,...,...,...
953883,S7157879,1678931180,2545011546,2181.0,1,2025-10-30
953890,D8257910,1637563662,1659022227,2108.0,1,2025-10-30
953903,17107230,1678985641,1678985487,530.0,1,2025-10-30
953909,S8855956,1083740944,2641683097,2375.0,1,2025-10-30


In [439]:
df_all=df_all.drop_duplicates(subset=['Дата','Артикул OZ'])

In [440]:
df_funnel.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'В корзину из поиска или каталога',
       'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Ozon ID', 'Расход, ₽',
       'Рекламные показы', 'Рекламные показы на карточке товара',
       'Рекламные заказано товаров', 'Рекламные заказано на сумму',
       'ТипАктивности', 'Цена'],
      dtype='object')

In [441]:
import numpy as np
import pandas as pd
import time

# --- настройки/утилиты ---
SPEND_CANDIDATES = ['Расход, ₽', 'Расход, руб', 'Расход, Р', 'Расход']

def _pick_spend_col(df: pd.DataFrame) -> str:
    for c in SPEND_CANDIDATES:
        if c in df.columns:
            return c
    raise KeyError(f"Не найдена колонка расхода среди: {SPEND_CANDIDATES}")

def to_money(s: pd.Series) -> pd.Series:
    """Чистит денежный столбец: NBSP/пробелы/₽, удаляет тысячные точки, заменяет запятую на точку."""
    ss = (s.astype(str)
            .str.strip()
            .str.replace('\u00A0', '', regex=False)   # NBSP
            .str.replace(' ',      '', regex=False)   # обычные пробелы
            .str.replace('₽',      '', regex=False))
    # 1.234,56 -> 1234,56 (сносим тысячные точки)
    ss = ss.str.replace(r'(?<=\d)\.(?=\d{3}(?:\D|$))', '', regex=True)
    # десятичная запятая -> точка
    ss = ss.str.replace(',', '.', regex=False)
    # оставляем только цифры/.-+eE
    ss = ss.str.replace(r'[^0-9\.\-\+eE]', '', regex=True)
    return pd.to_numeric(ss, errors='coerce')

# 10. Связать "Воронка" с "Справочник" БЕЗ дублирования и с сохранением расхода
try:
    print("Начинаем создавать таблицу ВоронкаСправочник...")
    start_time = time.time()

    # --- 0) Валидация входа ---
    required_columns = ["Дата", "Артикул"]
    for col in required_columns:
        if col not in df_funnel.columns:
            raise ValueError(f"Отсутствует столбец '{col}' в df_funnel.")

    # --- 1) Нормализация ключей одинаково в обеих таблицах ---
    df_funnel = df_funnel.copy()
    df_reference = df_reference.copy()

    df_funnel["Артикул"] = (df_funnel["Артикул"].fillna('')
                                              .astype(str).str.strip().str.upper()
                                              .str[:8])
    df_reference["Артикул"] = (df_reference["Артикул"].fillna('')
                                                    .astype(str).str.strip().str.upper()
                                                    .str[:8])
    # Дата (без времени)
    df_funnel["Дата"] = pd.to_datetime(df_funnel["Дата"], errors="coerce").dt.normalize()

    # --- 2) Денежный столбец: привести к числам ДО любых сумм/сравнений ---
    try:
        spend_col = _pick_spend_col(df_funnel)
        df_funnel[spend_col] = to_money(df_funnel[spend_col]).astype('float64')
    except KeyError:
        spend_col = None
        print("[INFO] В df_funnel нет колонки расхода — инварианты по расходу не проверяем.")

    # --- 3) Справочник: взять только нужные колонки и сделать уникальным по Артикулу ---
    reference_columns = [
        "Артикул", "Артикул OZ", "Наименование", "Коллекция", "Бренд", "Сезон", "Направление",
        "Розничный отдел", "Группа", "Бизнес-группа", "Техсегмент",
        "Байер", "Две последние коллекции", "Основной артикул", "Себестоимость с НДС",
        "Процент выкупа", "НДС", "Ответственный за группу", "Группа для отчетов"
    ]
    reference_columns = [c for c in reference_columns if c in df_reference.columns]
    df_reference_filtered = df_reference[["Артикул"] + [c for c in reference_columns if c != "Артикул"]].copy()

    dups = df_reference_filtered["Артикул"].duplicated(keep=False).sum()
    if dups:
        print(f"[INFO] В справочнике обнаружены дубликаты по 'Артикул' (после .str[:8]): {dups} строк.")

    ref_unique = (df_reference_filtered
                  .sort_values(["Артикул"])
                  .drop_duplicates(subset=["Артикул"], keep="first")
                  .reset_index(drop=True))
    assert not ref_unique["Артикул"].duplicated().any(), "ref_unique всё ещё содержит дубликаты Артикул"

    # --- 4) Контроль инвариантов расхода ДО merge ---
    if spend_col:
        before_total = pd.to_numeric(df_funnel[spend_col], errors='coerce').sum()
        before_by_date = (df_funnel.groupby("Дата", as_index=False)[spend_col]
                                   .sum(min_count=1)
                                   .rename(columns={spend_col: "Расход_до"}))

    # --- 5) LEFT-merge строго m:1 (никаких размножений) ---
    df_funnel_reference = pd.merge(
        df_funnel,
        ref_unique,
        on="Артикул",
        how="left",
        validate="m:1",
        indicator=False
    )

    # # --- 6) Контроль ПОСЛЕ merge (и принудительно в float64 перед isclose) ---
    # if spend_col:
    #     # На всякий случай — привести и после merge (если было форматирование где-то дальше)
    #     df_funnel_reference[spend_col] = pd.to_numeric(df_funnel_reference[spend_col], errors='coerce').astype('float64')

    #     after_total = df_funnel_reference[spend_col].sum()
    #     after_by_date = (df_funnel_reference.groupby("Дата", as_index=False)[spend_col]
    #                                       .sum(min_count=1)
    #                                       .rename(columns={spend_col: "Расход_после"}))

    #     check = before_by_date.merge(after_by_date, on="Дата", how="outer").fillna(0)

    #     # Гарантированно числовые типы для сравнения
    #     check["Расход_до"] = pd.to_numeric(check["Расход_до"], errors='coerce').astype('float64')
    #     check["Расход_после"] = pd.to_numeric(check["Расход_после"], errors='coerce').astype('float64')

    #     same = np.isclose(
    #         check["Расход_до"].to_numpy(dtype='float64'),
    #         check["Расход_после"].to_numpy(dtype='float64'),
    #         rtol=1e-9, atol=1e-6
    #     )
    #     drift = check.loc[~same]

    #     print(f"[CHECK] Общая сумма расхода: до={before_total:,.2f} | после={after_total:,.2f}")
    #     if not drift.empty:
    #         print("[WARN] Обнаружены расхождения по датам (первые 10):")
    #         print(drift.head(10))

    # --- 7) Итог ---
    print("Первые 5 строк таблицы ВоронкаСправочник:")
    print(df_funnel_reference.head())

    elapsed_time = time.time() - start_time
    print(f"Таблица ВоронкаСправочник успешно создана. Время выполнения: {elapsed_time:.2f} c.")

except Exception as e:
    print(f"Ошибка при создании таблицы ВоронкаСправочник: {e}")


Начинаем создавать таблицу ВоронкаСправочник...
[INFO] В справочнике обнаружены дубликаты по 'Артикул' (после .str[:8]): 114 строк.
Первые 5 строк таблицы ВоронкаСправочник:
        Дата   Артикул  Показы, всего  Показы на карточке товара  \
0 2025-11-09  00006000              2                          0   
1 2025-11-09  00006020              2                          0   
2 2025-11-09  00006080              1                          0   
3 2025-11-09  000060H0              1                          0   
4 2025-11-09  00006130              1                          0   

   Показы в поиске и каталоге  Позиция в поиске и каталоге  В корзину, всего  \
0                           0                          0.0                 0   
1                           0                          0.0                 0   
2                           0                          0.0                 0   
3                           0                          0.0                 0   
4                

In [442]:
df_funnel[df_funnel['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.int64(132002253)

In [443]:
df_funnel[df_funnel['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(8307669.5)

In [444]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(8307669.5)

In [445]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.int64(132002253)

In [446]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2025-11-09,00006000,2,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006000,402.6608,0.927632,20.0,Прусс Константин,Обувь
1,2025-11-09,00006020,2,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006020,382.3097,0.927632,20.0,Прусс Константин,Обувь
2,2025-11-09,00006080,1,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006080,408.8161,0.927632,20.0,Прусс Константин,Обувь
3,2025-11-09,000060H0,1,0,0,0.0,0,0,0,0,...,Обувь,flat (L),Коновалова А.,2019SS,000060H0,1017.1191,0.927632,20.0,Селютина Арина,Обувь
4,2025-11-09,00006130,1,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006130,377.0939,0.927632,20.0,Прусс Константин,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4364822,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4364823,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4364824,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4364825,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [447]:
df_funnel_reference['Расход, ₽'] = pd.to_numeric(
    df_funnel_reference['Расход, ₽']
        .astype(str)
        .str.replace('\u00A0', '', regex=False)  # NBSP
        .str.replace(' ',      '', regex=False)  # обычные пробелы
        .str.replace('₽',      '', regex=False)
        .str.replace(',',      '.', regex=False),  # ВАЖНО: str.replace, не replace
    errors='coerce'
)


In [448]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2025-11-09,00006000,2,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006000,402.6608,0.927632,20.0,Прусс Константин,Обувь
1,2025-11-09,00006020,2,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006020,382.3097,0.927632,20.0,Прусс Константин,Обувь
2,2025-11-09,00006080,1,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006080,408.8161,0.927632,20.0,Прусс Константин,Обувь
3,2025-11-09,000060H0,1,0,0,0.0,0,0,0,0,...,Обувь,flat (L),Коновалова А.,2019SS,000060H0,1017.1191,0.927632,20.0,Селютина Арина,Обувь
4,2025-11-09,00006130,1,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006130,377.0939,0.927632,20.0,Прусс Константин,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4364822,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4364823,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4364824,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4364825,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [449]:
df_union_agg#.drop_duplicates(subset=['Артикул OZ', 'SKU из объединенной карточки'])

,Артикул,Дата,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт"
0,00206110,2025-10-02,149484604,149484603,1706.0,2
1,00206110,2025-10-03,149484604,149484602,853.0,1
2,00206110,2025-10-05,149484604,149484603,1706.0,2
3,00206110,2025-10-06,149484604,149484603,853.0,1
4,00206110,2025-10-07,149484604,149484450,1363.0,1
...,...,...,...,...,...,...
131165,Y9708130,2025-11-05,1950209178,1979141782,1191.0,1
131166,Y9708130,2025-11-15,1950209178,1950208719,2081.0,2
131167,Y9708130,2025-11-20,1950209178,1950208649,1536.0,1
131168,Y9708130,2025-12-13,1950209178,1950208649,1536.0,1


In [450]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'В корзину из поиска или каталога',
       'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Ozon ID', 'Расход, ₽',
       'Рекламные показы', 'Рекламные показы на карточке товара',
       'Рекламные заказано товаров', 'Рекламные заказано на сумму',
       'ТипАктивности', 'Цена', 'Артикул OZ', 'Наименование', 'Коллекция',
       'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Группа',
       'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции',
       'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов'],
      dtype='object')

In [451]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2025-11-09,00006000,2,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006000,402.6608,0.927632,20.0,Прусс Константин,Обувь
1,2025-11-09,00006020,2,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006020,382.3097,0.927632,20.0,Прусс Константин,Обувь
2,2025-11-09,00006080,1,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006080,408.8161,0.927632,20.0,Прусс Константин,Обувь
3,2025-11-09,000060H0,1,0,0,0.0,0,0,0,0,...,Обувь,flat (L),Коновалова А.,2019SS,000060H0,1017.1191,0.927632,20.0,Селютина Арина,Обувь
4,2025-11-09,00006130,1,0,0,0.0,0,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,00006130,377.0939,0.927632,20.0,Прусс Константин,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4364822,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4364823,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4364824,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4364825,2025-12-25,,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [459]:
df_all['Дата'] = pd.to_datetime(df_all['Дата'])
# объединяем с df_funnel
df_result = df_funnel_reference.merge(
    df_all,
    on=["Дата", "Артикул OZ"],
    how="left"
)

df_union_agg['Дата'] = pd.to_datetime(df_union_agg['Дата'])
df_result = df_result.merge(
    df_union_agg.drop(columns=['Артикул']),
    on=["Дата", "Артикул OZ"],
    how="left"
)

# Трафарет
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') &
    (df_result['Место размещения'] == 'Поиск и рекомендации')) |
    (df_result['ТипАктивности'] == 'Оплата за клик')
)
df_result.loc[mask, 'ТипАктивности'] = 'Трафарет'

# Вывод в топ
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') & 
     (df_result['Место размещения'] == 'Поиск')) |
    (df_result['ТипАктивности'] == 'ТОП')
)
df_result.loc[mask, 'ТипАктивности'] = 'Вывод в топ'

# Органика
mask = (
    ((df_result['Расход, ₽'] == 0) | 
     (df_result['Расход, ₽'] == 0.0))
)
df_result.loc[mask, 'ТипАктивности'] = 'Органика'

In [460]:
df_union_agg

,Артикул,Дата,Артикул OZ,SKU из объединенной карточки,"Продажи, ₽","Заказы, шт"
0,00206110,2025-10-02,149484604,149484603,1706.0,2
1,00206110,2025-10-03,149484604,149484602,853.0,1
2,00206110,2025-10-05,149484604,149484603,1706.0,2
3,00206110,2025-10-06,149484604,149484603,853.0,1
4,00206110,2025-10-07,149484604,149484450,1363.0,1
...,...,...,...,...,...,...
131165,Y9708130,2025-11-05,1950209178,1979141782,1191.0,1
131166,Y9708130,2025-11-15,1950209178,1950208719,2081.0,2
131167,Y9708130,2025-11-20,1950209178,1950208649,1536.0,1
131168,Y9708130,2025-12-13,1950209178,1950208649,1536.0,1


In [462]:
df_result[df_result['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.int64(132002253)

In [463]:
df_result[df_result['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(8307669.5)

In [465]:
df_result['ТипАктивности'].unique()

array([nan, 'Трафарет', 'Органика', 'Вывод в топ',
       'Оплата за заказ и Оплата за клик', 'Оплата за заказ', 'Трафареты',
       'Спецразмещение', 'Оплата за заказ и Спецразмещение и Трафареты',
       'Оплата за заказ и Трафареты', 'Вывод в топ и Трафареты',
       'Спецразмещение и Трафареты', 'Оплата за заказ и Спецразмещение',
       'Вывод в топ и Оплата за заказ'], dtype=object)

In [466]:
df_result.rename(columns={'Продажи, ₽':'Ассоциированные заказы, руб', 'Заказы, шт':'Ассоциированные заказы, шт'}, inplace=True)

In [467]:
# df_funnel_reference_copy = df_funnel_reference.copy()
df_funnel_reference = df_result

In [468]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(8307669.5)

In [470]:
funnel_columns = ['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена'
       ]
funnel_columns_all = [
    'Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
    'Показы на карточке товара', 'Показы в поиске и каталоге',
    'Позиция в поиске и каталоге', 'В корзину, всего',
    'Заказано товаров', 'Отменено товаров', 'Доставлено товаров',
    'Возвращено товаров', 'Заказано на сумму',
    'В корзину из карточки товара', 'Выкупили ШТ',
    'Расход, ₽', 'Рекламные заказано на сумму',
    'Рекламные заказано товаров', 'Рекламные показы',
    'Рекламные показы на карточке товара', 'Цена',
    'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
    'Направление', 'Розничный отдел', 'Модель', 'Группа',
    'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции',
    'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС',
    'Ответственный за группу', 'Группа для отчетов',
    'ID кампании', 'Инструмент', 'Место размещения'
]
funnel_columns_widing = ['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена'
       ]

In [471]:
# import pandas as pd
# import numpy as np

# # --- 0) подготовка
# df = df_funnel_reference.copy()

# # переименовать тип активности "ТОП" -> "Вывод в топ"
# df.loc[df['ТипАктивности'].eq('ТОП'), 'ТипАктивности'] = 'Вывод в топ'

# # полный список типов (жёстко фиксируем порядок и наличие)
# ALL_TYPES = ['Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Спецразмещение', 'Органика']

# # ключи и метрики
# key_cols = ['Дата', 'Артикул']
# # всё, что стоит в таблице ПОСЛЕ "ТипАктивности", считаем метриками
# cols = funnel_columns
# metric_cols = cols[cols.index('ТипАктивности') + 1 :]

# print(metric_cols)

# # привести метрики к числам (на случай строк/пробелов)
# for c in metric_cols:
#     df[c] = pd.to_numeric(df[c], errors='coerce')

# # --- 1) агрегация по (Дата, Артикул, ТипАктивности)
# g = (df
#      .groupby(key_cols + ['ТипАктивности'], as_index=False)[metric_cols]
#      .sum(min_count=1)
# )

# # --- 2) «широкая» таблица метрик с префиксами <Тип>_<Метрика>
# wide_metrics = g.pivot_table(
#     index=key_cols,
#     columns='ТипАктивности',
#     values=metric_cols,
#     aggfunc='sum',
#     fill_value=0
# )

# # гарантируем наличие ВСЕХ типов и ВСЕХ метрик (даже если их не было в данных)
# full_cols = pd.MultiIndex.from_product([metric_cols, ALL_TYPES])
# wide_metrics = wide_metrics.reindex(columns=full_cols, fill_value=0)

# # имена колонок: "Тип_Метрика"
# wide_metrics.columns = [f'{act}_{met}' for met, act in wide_metrics.columns.to_flat_index()]
# wide_metrics = wide_metrics.reset_index()

# # --- 3) бинарные признаки наличия типа активности (1/0) по каждой паре (Дата, Артикул)
# presence = (
#     df.groupby(key_cols + ['ТипАктивности']).size()
#       .reset_index(name='n')
#       .pivot(index=key_cols, columns='ТипАктивности', values='n')
#       .reindex(columns=ALL_TYPES, fill_value=0)
#       .gt(0).astype(int)  # 1 если был хотя бы один ряд данного типа
#       .reset_index()
# )

# # --- 4) объединяем метрики и бинарные признаки
# out = (wide_metrics
#        .merge(presence, on=key_cols, how='left')
#        .fillna(0)
# )

# # --- 5) итоговые столбцы БЕЗ префиксов = сумма по всем типам
# for met in metric_cols:
#     to_sum = [f'{t}_{met}' for t in ALL_TYPES if f'{t}_{met}' in out.columns]
#     if to_sum:
#         out[met] = out[to_sum].sum(axis=1)

# # --- 6) порядок колонок: ключи → бинарные типы → для каждой метрики столбцы по типам → итог по метрике
# ordered = key_cols + ALL_TYPES[:]  # бинарные столбцы имеют те же имена, что и типы
# for met in metric_cols:
#     ordered += [f'{t}_{met}' for t in ALL_TYPES]
#     ordered += [met]
# # оставим только реально существующие (вдруг каких-то метрик не было)
# ordered = [c for c in ordered if c in out.columns]

# out = out[ordered]

# # результат в переменной `out`
# # одна строка на (Дата, Артикул), колоноки вида:
# # Дата | Артикул | Вывод в топ | Трафарет | ... | Вывод в топ_Показы, всего | Трафарет_Показы, всего | ... | Показы, всего | ...


In [473]:
import numpy as np
import pandas as pd

def build_funnel_wide(
    df_raw: pd.DataFrame,
    funnel_columns: list,
    all_types=('Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика'),
    infer_organic_by_zero_spend=False,
    spend_col='Расход, ₽',
    extra_agg='first',          # 'first' | 'join'
    extra_join_sep=' | ',
    check_spend_invariance=True,
    atol=1e-6, rtol=1e-9
):
    """
    Склеивает строки по (Дата, Артикул), раскладывает метрики по типам активности,
    добавляет ИТОГИ, которые считаются напрямую из исходника по ключу (Дата, Артикул).
    Благодаря этому 'Расход, ₽' (и прочие итоги) сохраняют исходные значения.
    """

    # ---- 0) Исходная выборка для расчётов (только нужные колонки) ----
    cols_present = [c for c in funnel_columns if c in df_raw.columns]
    df = df_raw.loc[:, cols_present].copy()

    # ---- 1) Нормализуем тип активности ----
    type_col = 'ТипАктивности'
    df[type_col] = df[type_col].replace({'ТОП': 'Вывод в топ'})
    if infer_organic_by_zero_spend and spend_col in df.columns:
        m0 = pd.to_numeric(df[spend_col], errors='coerce').fillna(0).eq(0)
        df.loc[m0, type_col] = 'Органика'

    # ---- 2) Ключи/метрики и приведение типов ----
    key_cols = ['Дата', 'Артикул']
    met_start = funnel_columns.index(type_col) + 1
    metric_cols = [c for c in funnel_columns[met_start:] if c in df.columns]

    # аккуратно приводим метрики
    for c in metric_cols:
        if c == spend_col:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')   # спенд в float64
        else:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

    # ---- 3) ИТОГИ БЕЗ ПРЕФИКСОВ (ИСТИНА) по (Дата, Артикул) ----
    totals_df = (df.groupby(key_cols, as_index=False)[metric_cols]
                   .sum(min_count=1))   # если где-то все NaN, останется NaN; это корректно

    # ---- 4) Префиксные метрики по типам ----
    g = (df.groupby(key_cols + [type_col], as_index=False)[metric_cols]
           .sum(min_count=1))

    # Дополним all_types тем, что реально встретилось
    types_present = g[type_col].dropna().unique().tolist()
    all_types = list(dict.fromkeys(list(all_types) + [t for t in types_present if t not in all_types]))

    # Полная база ключей = все пары (Дата, Артикул), которые встречаются в исходнике
    base = (df[key_cols].drop_duplicates()
                    .set_index(key_cols)
                    .sort_index())

    # Сформируем блоки префиксных метрик и флаги наличия типов
    metric_blocks, flag_blocks = [], []
    for t in all_types:
        sub = g[g[type_col] == t].set_index(key_cols)

        if sub.empty:
            # пустой тип → нули на всю базу
            sub_metrics = pd.DataFrame(
                0.0, index=base.index,
                columns=[f'{t}_{m}' for m in metric_cols],
                dtype='float32'
            )
        else:
            sub_metrics = (sub[metric_cols]
                           .rename(columns={m: f'{t}_{m}' for m in metric_cols})
                           .reindex(base.index, fill_value=0.0))

            # типы данных: спенд оставляем float64
            for col in sub_metrics.columns:
                if col.endswith(spend_col):
                    sub_metrics[col] = sub_metrics[col].astype('float64')
                else:
                    sub_metrics[col] = sub_metrics[col].astype('float32')

        metric_blocks.append(sub_metrics)

        # бинарный флаг присутствия типа (на уровне ключа)
        flag = pd.Series(1, index=sub.index, name=t) if not sub.empty else pd.Series(0, index=base.index, name=t)
        flag_blocks.append(flag.reindex(base.index, fill_value=0).astype('int8'))

    metrics_block = pd.concat(metric_blocks, axis=1)
    flags_block   = pd.concat(flag_blocks, axis=1)

    # ---- 5) СБОРКА CORE: ключи + флаги + префиксные метрики + ИТОГИ ИЗ totals_df ----
    core = pd.concat(
        [
            base.reset_index(),
            flags_block.reset_index(drop=True),
            metrics_block.reset_index(drop=True)
        ],
        axis=1
    )

    # присоединяем ИТОГИ (истина) строго m:1
    core = core.merge(totals_df, on=key_cols, how='left', validate='m:1')

    # ---- 6) ДОП. колонки из df_raw (НЕ участвуют в расчётах) ----
    exclude = set(key_cols + [type_col] + metric_cols)
    extra_cols = [c for c in df_raw.columns if c not in exclude]
    if extra_cols:
        if extra_agg == 'first':
            dims_block = (df_raw[key_cols + extra_cols]
                            .sort_values(key_cols)
                            .groupby(key_cols, as_index=False)
                            .first())
        elif extra_agg == 'join':
            def _join_unique(s):
                v = pd.unique(s.dropna().astype(str))
                return extra_join_sep.join(v) if len(v) else np.nan
            dims_block = (df_raw[key_cols + extra_cols]
                            .groupby(key_cols, as_index=False)
                            .agg({c: _join_unique for c in extra_cols}))
        else:
            raise ValueError("extra_agg должен быть 'first' или 'join'")

        out = core.merge(dims_block, on=key_cols, how='left', validate='m:1')
    else:
        out = core

    # ---- 7) Проверка инварианта для 'Расход, ₽' (опционально) ----
    if check_spend_invariance and (spend_col in totals_df.columns):
        base_sp = (totals_df.groupby(key_cols, as_index=False)[spend_col].sum(min_count=1)
                             .rename(columns={spend_col: '__base__'}))
        after_sp = (out.groupby(key_cols, as_index=False)[spend_col].sum(min_count=1)
                          .rename(columns={spend_col: '__after__'}))
        chk = base_sp.merge(after_sp, on=key_cols, how='outer').fillna(0)
        bad = chk.loc[~np.isclose(chk['__base__'], chk['__after__'], rtol=rtol, atol=atol)]
        if not bad.empty:
            print("[WARN] Инвариант по 'Расход, ₽' нарушен для некоторых ключей (первые 10):")
            print(bad.head(10))

    # ---- 8) Порядок колонок: ключи → доп.колонки → флаги → префиксные метрики → ИТОГИ ----
    ordered = []
    ordered += key_cols
    ordered += [c for c in df_raw.columns if (c in out.columns and c not in key_cols and c not in ([type_col] + metric_cols))]
    ordered += [t for t in all_types if t in out.columns]

    for m in metric_cols:
        # префиксные
        ordered += [f'{t}_{m}' for t in all_types if f'{t}_{m}' in out.columns]
    # ИТОГИ (без префикса) — в самом конце блоком в исходном порядке
    ordered += [m for m in metric_cols if m in out.columns]

    out = out[[c for c in ordered if c in out.columns]].copy()
    return out

In [474]:
final_after_widing_columns = ['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'ID кампании', 'Инструмент', 'Место размещения', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталоге', 'Трафарет_Позиция в поиске и каталоге', 'Оплата за заказ_Позиция в поиске и каталоге', 'Органика_Позиция в поиске и каталоге', 'Вывод в топ_В корзину, всего', 'Трафарет_В корзину, всего', 'Оплата за заказ_В корзину, всего', 'Органика_В корзину, всего', 'Вывод в топ_Заказано товаров', 'Трафарет_Заказано товаров', 'Оплата за заказ_Заказано товаров', 'Органика_Заказано товаров', 'Вывод в топ_Отменено товаров', 'Трафарет_Отменено товаров', 'Оплата за заказ_Отменено товаров', 'Органика_Отменено товаров', 'Вывод в топ_Доставлено товаров', 'Трафарет_Доставлено товаров', 'Оплата за заказ_Доставлено товаров', 'Органика_Доставлено товаров', 'Вывод в топ_Возвращено товаров', 'Трафарет_Возвращено товаров', 'Оплата за заказ_Возвращено товаров', 'Органика_Возвращено товаров', 'Вывод в топ_Заказано на сумму', 'Трафарет_Заказано на сумму', 'Оплата за заказ_Заказано на сумму', 'Органика_Заказано на сумму', 'Вывод в топ_В корзину из карточки товара', 'Трафарет_В корзину из карточки товара', 'Оплата за заказ_В корзину из карточки товара', 'Органика_В корзину из карточки товара', 'Вывод в топ_Выкупили ШТ', 'Трафарет_Выкупили ШТ', 'Оплата за заказ_Выкупили ШТ', 'Органика_Выкупили ШТ', 'Вывод в топ_Расход, ₽', 'Трафарет_Расход, ₽', 'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽', 'Вывод в топ_Рекламные заказано на сумму', 'Трафарет_Рекламные заказано на сумму', 'Оплата за заказ_Рекламные заказано на сумму', 'Органика_Рекламные заказано на сумму', 'Вывод в топ_Рекламные заказано товаров', 'Трафарет_Рекламные заказано товаров', 'Оплата за заказ_Рекламные заказано товаров', 'Органика_Рекламные заказано товаров', 'Вывод в топ_Рекламные показы', 'Трафарет_Рекламные показы', 'Оплата за заказ_Рекламные показы', 'Органика_Рекламные показы', 'Вывод в топ_Рекламные показы на карточке товара', 'Трафарет_Рекламные показы на карточке товара', 'Оплата за заказ_Рекламные показы на карточке товара', 'Органика_Рекламные показы на карточке товара', 'Вывод в топ_Цена', 'Трафарет_Цена', 'Оплата за заказ_Цена', 'Органика_Цена', 'Показы, всего', 'Показы на карточке товара', 'Показы в поиске и каталоге', 'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров', 'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму', 'В корзину из карточки товара', 'Выкупили ШТ', 'Расход, ₽', 'Рекламные заказано на сумму', 'Рекламные заказано товаров', 'Рекламные показы', 'Рекламные показы на карточке товара', 'Цена']

In [475]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'В корзину из поиска или каталога',
       'Выкупили ШТ', 'Тип товара', 'Товары', 'Модель', 'Ozon ID', 'Расход, ₽',
       'Рекламные показы', 'Рекламные показы на карточке товара',
       'Рекламные заказано товаров', 'Рекламные заказано на сумму',
       'ТипАктивности', 'Цена', 'Артикул OZ', 'Наименование', 'Коллекция',
       'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Группа',
       'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции',
       'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов', 'ID кампании',
       'Инструмент', 'Место размещения', 'SKU из объединенной карточ

In [476]:
out = build_funnel_wide(df_raw=df_funnel_reference, funnel_columns=funnel_columns_widing)
out = out[final_after_widing_columns]
out

,Дата,Артикул,Артикул OZ,Наименование,Коллекция,Бренд,Сезон,Направление,Розничный отдел,Модель,...,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,Выкупили ШТ,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Цена
0,2025-08-01,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,7144419.87,NaN,4713.0,35078956.0,672826.0,NaN
1,2025-10-01,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,5639637.94,NaN,4540.0,34058968.0,807386.0,NaN
2,2025-10-02,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,7044779.85,NaN,4398.0,43960896.0,1081493.0,NaN
3,2025-10-03,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,6393859.26,NaN,5088.0,42768960.0,1077165.0,NaN
4,2025-10-04,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,6207978.31,NaN,4270.0,39605616.0,977044.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332144,2025-12-25,Y9808110,1997933020,Шорты мужские A85512-2,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,Шорты Kari мужские спортивные,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2332145,2025-12-25,Y9808120,1997933206,Шорты мужские A85512-3,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,Шорты Kari мужские спортивные,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2332146,2025-12-25,Y9808130,1934644792,Шорты мужские SS25C2020,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",Шорты Kari мужские спортивные,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2332147,2025-12-25,Y9808140,1950207731,Шорты мужские SS25C2021,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",Шорты Kari мужские спортивные,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [479]:
print(len(list(out.columns)))

121


In [480]:
print(list(out.columns))

['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'ID кампании', 'Инструмент', 'Место размещения', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталог

In [481]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(8307669.5)

In [482]:
out[out['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(8307669.500000001)

In [483]:
df_funnel_reference[df_funnel_reference['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.int64(132002253)

In [484]:
out[out['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.float32(132002240.0)

In [485]:
df_funnel_reference = out

In [486]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд',
       'Сезон', 'Направление', 'Розничный отдел', 'Модель',
       ...
       'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 'Расход, ₽',
       'Рекламные заказано на сумму', 'Рекламные заказано товаров',
       'Рекламные показы', 'Рекламные показы на карточке товара', 'Цена'],
      dtype='object', length=121)

In [487]:
df_funnel_reference

,Дата,Артикул,Артикул OZ,Наименование,Коллекция,Бренд,Сезон,Направление,Розничный отдел,Модель,...,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,Выкупили ШТ,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Цена
0,2025-08-01,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,7144419.87,NaN,4713.0,35078956.0,672826.0,NaN
1,2025-10-01,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,5639637.94,NaN,4540.0,34058968.0,807386.0,NaN
2,2025-10-02,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,7044779.85,NaN,4398.0,43960896.0,1081493.0,NaN
3,2025-10-03,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,6393859.26,NaN,5088.0,42768960.0,1077165.0,NaN
4,2025-10-04,,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,6207978.31,NaN,4270.0,39605616.0,977044.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332144,2025-12-25,Y9808110,1997933020,Шорты мужские A85512-2,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,Шорты Kari мужские спортивные,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2332145,2025-12-25,Y9808120,1997933206,Шорты мужские A85512-3,2025SS,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,Шорты Kari мужские спортивные,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2332146,2025-12-25,Y9808130,1934644792,Шорты мужские SS25C2020,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",Шорты Kari мужские спортивные,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2332147,2025-12-25,Y9808140,1950207731,Шорты мужские SS25C2021,2025SS,kari,"весна, лето",Одежда для мужчин,"Брюки, шорты мужские",Шорты Kari мужские спортивные,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [488]:
# 11. Связать "ВоронкаСправочник" с "Остатки с дистрибуцией"
try:
    print("Начинаем создавать таблицу ДБбезПризнаков...")
    start_time = time.time()  # Запускаем таймер
    df_stock_with_distribution['Дата'] = pd.to_datetime(df_stock_with_distribution['Дата'])
    df_final_db = pd.merge(df_funnel_reference, df_stock_with_distribution, left_on=["Дата", "Артикул"], right_on=["Дата", "Артикул"], how="left")
    df_final_db = format_date_column(df_final_db, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБбезПризнаков:")
    print(df_final_db.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБбезПризнаков успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБбезПризнаков: {e}")

Начинаем создавать таблицу ДБбезПризнаков...
Первые 5 строк таблицы ДБбезПризнаков:
         Дата Артикул Артикул OZ Наименование Коллекция Бренд Сезон  \
0  2025-08-01               None         None      None  None  None   
1  2025-10-01               None         None      None  None  None   
2  2025-10-02               None         None      None  None  None   
3  2025-10-03               None         None      None  None  None   
4  2025-10-04               None         None      None  None  None   

  Направление Розничный отдел Модель  ... В корзину из карточки товара  \
0        None            None   None  ...                          NaN   
1        None            None   None  ...                          NaN   
2        None            None   None  ...                          NaN   
3        None            None   None  ...                          NaN   
4        None            None   None  ...                          NaN   

  Выкупили ШТ   Расход, ₽ Рекламные заказано

In [489]:
df_final_db.columns

Index(['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд',
       'Сезон', 'Направление', 'Розничный отдел', 'Модель',
       ...
       'В корзину из карточки товара', 'Выкупили ШТ', 'Расход, ₽',
       'Рекламные заказано на сумму', 'Рекламные заказано товаров',
       'Рекламные показы', 'Рекламные показы на карточке товара', 'Цена',
       'Остаток Агрегатора', 'Дистрибуция'],
      dtype='object', length=123)

In [490]:
# 5. Получить данные из файла !!!_Признаки для артикула и даты для Озон
try:
    print("Начинаем получать данные для Признаков...")
    start_time = time.time()  # Запускаем таймер
    file_path_features = os.path.join(FOLDER_PATH_FEATURES, "!!!_Признаки для артикула и даты для Озон.xlsx")
    if os.path.exists(file_path_features):
        df_item_features = pd.read_excel(file_path_features, sheet_name="Признаки для артикула", dtype=str, engine="calamine")
        df_date_features = pd.read_excel(file_path_features, sheet_name="Признаки для дат", dtype={0: "datetime64[ns]", **{i: str for i in range(1, 6)}}, engine="calamine")

        # Обработка ошибок
        df_item_features = handle_errors(df_item_features)
        df_date_features = handle_errors(df_date_features)

        # Форматирование даты
        df_date_features = format_date_column(df_date_features, 'Дата')

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки артикула:")
        print(df_item_features.head())

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки дат:")
        print(df_date_features.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Признаков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл '!!!_Признаки для артикула и даты для Озон.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Признаков: {e}")

Начинаем получать данные для Признаков...
Первые 5 строк таблицы Признаки артикула:
    Артикул Признак Артикула 1 Признак Артикула 2 Признак Артикула 3  \
0  00001851                NaN                NaN                NaN   
1  00001852                NaN                NaN                NaN   
2  00001855                NaN                NaN                NaN   
3  00001856                NaN                NaN                NaN   
4  00001931                NaN                NaN                NaN   

  Признак Артикула 4 Признак Артикула 5  
0                NaN                NaN  
1                NaN                NaN  
2                NaN                NaN  
3                NaN                NaN  
4                NaN                NaN  
Первые 5 строк таблицы Признаки дат:
Empty DataFrame
Columns: [Дата, Признак Даты 1, Признак Даты 2, Признак Даты 3, Признак Даты 4, Признак Даты 5]
Index: []
Данные для Признаков успешно сохранены. Время выполнения: 0 часа(ов) 0 м

In [491]:
# 12. Связать "ДБбезПризнаков" с "Признаки для артикула"
try:
    print("Начинаем создавать таблицу ДБсПризнакамиАртикула...")
    start_time = time.time()  # Запускаем таймер
    df_final_db_item_features = pd.merge(df_final_db, df_item_features, left_on=["Артикул"], right_on=["Артикул"], how="left")
    df_final_db_item_features = format_date_column(df_final_db_item_features, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБсПризнакамиАртикула:")
    print(df_final_db_item_features.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБсПризнакамиАртикула успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнакамиАртикула: {e}")

Начинаем создавать таблицу ДБсПризнакамиАртикула...
Первые 5 строк таблицы ДБсПризнакамиАртикула:
         Дата Артикул Артикул OZ Наименование Коллекция Бренд Сезон  \
0  2025-08-01               None         None      None  None  None   
1  2025-10-01               None         None      None  None  None   
2  2025-10-02               None         None      None  None  None   
3  2025-10-03               None         None      None  None  None   
4  2025-10-04               None         None      None  None  None   

  Направление Розничный отдел Модель  ... Рекламные показы  \
0        None            None   None  ...       35078956.0   
1        None            None   None  ...       34058968.0   
2        None            None   None  ...       43960896.0   
3        None            None   None  ...       42768960.0   
4        None            None   None  ...       39605616.0   

  Рекламные показы на карточке товара Цена Остаток Агрегатора Дистрибуция  \
0                        

In [492]:
# 13. Связать "ДБсПризнакамиАртикула" с "Признаки для дат"
try:
    print("Начинаем создавать таблицу ДБсПризнаками...")
    start_time = time.time()
    df_final_db_all_features = pd.merge(df_final_db_item_features, df_date_features, on="Дата", how="left")
    df_final_db_all_features = format_date_column(df_final_db_all_features, 'Дата')

    print("Первые 5 строк таблицы ДБсПризнаками:")
    print(df_final_db_all_features.head())

    # Сохранение финальной таблицы
    # df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon_New.csv"), index=False)

    elapsed_time = time.time() - start_time
    print(f"Таблица ДБсПризнаками успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнаками: {e}")

Начинаем создавать таблицу ДБсПризнаками...
Первые 5 строк таблицы ДБсПризнаками:
         Дата Артикул Артикул OZ Наименование Коллекция Бренд Сезон  \
0  2025-08-01               None         None      None  None  None   
1  2025-10-01               None         None      None  None  None   
2  2025-10-02               None         None      None  None  None   
3  2025-10-03               None         None      None  None  None   
4  2025-10-04               None         None      None  None  None   

  Направление Розничный отдел Модель  ... Признак Артикула 1  \
0        None            None   None  ...                NaN   
1        None            None   None  ...                NaN   
2        None            None   None  ...                NaN   
3        None            None   None  ...                NaN   
4        None            None   None  ...                NaN   

  Признак Артикула 2 Признак Артикула 3 Признак Артикула 4 Признак Артикула 5  \
0                NaN     

In [493]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '2025-11-17']['Расход, ₽'].sum()

np.float64(12742959.61)

In [494]:
import numpy as np
# df — ваша широкая таблица с флагами типов (1/0)
type_order_paid = ['Вывод в топ', 'Трафарет', 'Оплата за заказ']
paid_cols = [c for c in type_order_paid if c in df_final_db_all_features.columns]  # на случай отсутствующих

# Матрица флагов платных типов
flags = df_final_db_all_features[paid_cols].fillna(0).astype('uint8').to_numpy()
labels = np.array(paid_cols, dtype=object)

# Собираем подписи для платных комбинаций
combo = ['/'.join(labels[row.astype(bool)]) if row.any() else '' for row in flags]
df_final_db_all_features['ТипАктивности'] = combo

# Если есть только органика — подставим "Органика"
if 'Органика' in df_final_db_all_features.columns:
    only_org = df_final_db_all_features['Органика'].fillna(0).astype('uint8').eq(1) & (flags.sum(axis=1) == 0)
    df_final_db_all_features.loc[only_org, 'ТипАктивности'] = 'Органика'

# Пустые — на "—"
df_final_db_all_features['ТипАктивности'] = df_final_db_all_features['ТипАктивности'].replace('', '—')

In [495]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '2025-12-01']['Расход, ₽'].sum()

np.float64(8307669.500000001)

In [496]:
print(list(df_final_db_all_features.columns))

['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'ID кампании', 'Инструмент', 'Место размещения', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталог

In [497]:
df_final_db_all_features['Дата'].unique()

array(['2025-08-01', '2025-10-01', '2025-10-02', '2025-10-03',
       '2025-10-04', '2025-10-05', '2025-10-06', '2025-10-07',
       '2025-10-08', '2025-10-09', '2025-10-10', '2025-10-11',
       '2025-10-12', '2025-10-13', '2025-10-14', '2025-10-15',
       '2025-10-16', '2025-10-17', '2025-10-18', '2025-10-19',
       '2025-10-20', '2025-10-21', '2025-10-22', '2025-10-23',
       '2025-10-24', '2025-10-25', '2025-10-26', '2025-10-27',
       '2025-10-28', '2025-10-29', '2025-10-30', '2025-10-31',
       '2025-11-01', '2025-11-02', '2025-11-03', '2025-11-04',
       '2025-11-05', '2025-11-06', '2025-11-07', '2025-11-08',
       '2025-11-09', '2025-11-10', '2025-11-11', '2025-11-12',
       '2025-11-13', '2025-11-14', '2025-11-15', '2025-11-16',
       '2025-11-17', '2025-11-18', '2025-11-19', '2025-11-20',
       '2025-11-21', '2025-11-22', '2025-11-23', '2025-11-24',
       '2025-11-25', '2025-11-26', '2025-11-27', '2025-11-28',
       '2025-11-29', '2025-11-30', '2025-12-01', '2025-

In [498]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '2025-12-01']['Показы, всего'].sum()

np.float32(132002240.0)

In [499]:
# === SQL СЦЕПКИ ОЗОН ===
sql = """
SELECT scepka.[id]
      ,scepka.[offer_id]
      ,scepka.[product_id]
      ,sku.fbo_sku as [Артикул OZ]
      ,scepka.[group_value] as [Текущая склейка]
      ,sku.[article]
      ,scepka.[updated_at] as [Дата Обновления]
  FROM [DBReport].[mp].[ozon_scepka] scepka
  JOIN [DBReport].[mp].[ozon_sku] sku 
  ON  scepka.[product_id] = sku.[product_id] 
  and sku.actual = 1
"""
df_links = pd.read_sql(sql, engine)
df_links['Артикул OZ'] = df_links['Артикул OZ'].astype(str)
df_links.to_excel(os.path.join(FOLDER_PATH, f"Склейки товаров\\OZ\\{df_links['Дата Обновления'].iloc[0].strftime('%d.%m.%Y')}_Склейка Товаров_OZ.xlsx"))

In [500]:
df_links['Текущая склейка'].unique()

array(['W8429001', '1716', '862', ..., '68000050', '67700060', '68004000'],
      shape=(30697,), dtype=object)

In [501]:
print(list(df_final_db_all_features.columns))

['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'ID кампании', 'Инструмент', 'Место размещения', 'SKU из объединенной карточки', 'Ассоциированные заказы, руб', 'Ассоциированные заказы, шт', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталог

In [502]:
df_final_db_all_features = pd.merge(df_final_db_all_features, df_links[["Артикул OZ", "Текущая склейка"]], how='left', on='Артикул OZ')

In [503]:
df_final_db_all_features[df_final_db_all_features['Дата'] == '01.12.2025']['Расход, ₽'].sum()

np.float64(0.0)

In [504]:
df_final_db_all_features['Дата'] = pd.to_datetime(df_final_db_all_features['Дата'], format='%Y-%m-%d', errors='coerce').dt.strftime('%d.%m.%Y')

In [505]:
count = 0
for item in list(df_final_db_all_features.columns):
    print(f'{{"{item}", type {str(type(df_final_db_all_features[item].unique()[0]))}}}, ', end="")
    count +=1
    if count == 5:
        print("\n", end="")
        count = 0

{"Дата", type <class 'str'>}, {"Артикул", type <class 'str'>}, {"Артикул OZ", type <class 'NoneType'>}, {"Наименование", type <class 'NoneType'>}, {"Коллекция", type <class 'NoneType'>}, 
{"Бренд", type <class 'NoneType'>}, {"Сезон", type <class 'NoneType'>}, {"Направление", type <class 'NoneType'>}, {"Розничный отдел", type <class 'NoneType'>}, {"Модель", type <class 'NoneType'>}, 
{"Группа", type <class 'NoneType'>}, {"Бизнес-группа", type <class 'NoneType'>}, {"Техсегмент", type <class 'NoneType'>}, {"Байер", type <class 'NoneType'>}, {"Две последние коллекции", type <class 'NoneType'>}, 
{"Основной артикул", type <class 'NoneType'>}, {"Себестоимость с НДС", type <class 'numpy.float64'>}, {"Процент выкупа", type <class 'numpy.float64'>}, {"НДС", type <class 'numpy.float64'>}, {"Ответственный за группу", type <class 'NoneType'>}, 
{"Группа для отчетов", type <class 'NoneType'>}, {"ID кампании", type <class 'numpy.float64'>}, {"Инструмент", type <class 'NoneType'>}, {"Место размещения

In [506]:
print(len(list(df_final_db_all_features.columns)))

135


In [507]:
df_final_db_all_features.columns

Index(['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд',
       'Сезон', 'Направление', 'Розничный отдел', 'Модель',
       ...
       'Признак Артикула 3', 'Признак Артикула 4', 'Признак Артикула 5',
       'Признак Даты 1', 'Признак Даты 2', 'Признак Даты 3', 'Признак Даты 4',
       'Признак Даты 5', 'ТипАктивности', 'Текущая склейка'],
      dtype='object', length=135)

In [508]:
# Сохранение финальной таблицы
import pyarrow as pa
import pyarrow.csv as csv
table = pa.Table.from_pandas(df_final_db_all_features)
csv.write_csv(table, os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon_Test.csv"))
# df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon.csv"), index=False)

In [ ]:
# del df_date_features
# gc.collect()

In [4]:
# Функция для обновления Excel-файла с циклом попыток
def update_and_save_excel(file_path, new_file_path):
    max_attempts = 10  # Максимальное количество попыток
    attempt = 0

    while attempt < max_attempts:
        attempt += 1
        print(f"Попытка {attempt} обновить файл '{os.path.basename(file_path)}'...")

        try:
            # Открываем Excel приложение
            excel = win32.Dispatch("Excel.Application")
            excel.DisplayAlerts = False  # Отключает предупреждения Excel

            try:
                # Открываем книгу
                workbook = excel.Workbooks.Open(file_path)

                # Выполняем обновление всех данных (эквивалентно "Обновить всё" в Excel)
                print("Выполняем обновление данных...")
                workbook.RefreshAll()
                excel.CalculateUntilAsyncQueriesDone()  # Дожидаемся завершения обновления

                # Сохраняем оригинальный файл в FOLDER_PATH_FOR_DB
                workbook.SaveAs(file_path)
                print(f"Файл успешно сохранен с оригинальным именем в '{os.path.dirname(file_path)}'.")

                # Сохраняем файл с новым именем в FOLDER_PATH_FEATURES
                workbook.SaveAs(new_file_path)
                print(f"Файл успешно сохранен как '{os.path.basename(new_file_path)}'.")

                return True  # Успешное завершение

            except Exception as e:
                print(f"Ошибка при обновлении или сохранении файла: {e}")
            finally:
                # Закрываем книгу и выходим из Excel
                if 'workbook' in locals():
                    workbook.Close(SaveChanges=False)
                excel.Quit()

        except Exception as e:
            print(f"Ошибка при работе с Excel: {e}")

        # Если произошла ошибка, ждем перед следующей попыткой
        if attempt < max_attempts:
            print(f"Пауза перед следующей попыткой ({attempt + 1}/{max_attempts})...")
            time.sleep(60)  # Пауза 5 секунд

    return False  # Все попытки завершились неудачно

In [ ]:
# 19. Обновить файл "Показы и затраты ОЗ_2.0.xlsx"
try:
    print("Подготовка данных для ДБ завершена.")
    # input("Начать обновление файлов ДБ? Для подтверждения нажмите Enter...")
    print("Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...")
    start_time = time.time()  # Запускаем таймер

    # Путь к исходному файлу
    file_path_shows_expenses = os.path.join(FOLDER_PATH_FOR_DB, "Показы и затраты ОЗ_2.0_Test.xlsx")

    if os.path.exists(file_path_shows_expenses):
        # Создаем новое имя файла с текущей датой без года
        current_month_day = time.strftime("%d.%m")  # Текущая дата в формате ДД.ММ
        new_file_name = f"Показы и затраты ОЗ_2.0 {current_month_day}.xlsx"
        new_file_path = os.path.join(FOLDER_PATH_FEATURES, new_file_name)

        # Путь для сохранения в дополнительную папку FOLDER_PATH_DUDL
        dudl_file_path = os.path.join(FOLDER_PATH_DUDL, new_file_name)

        # Удаляем старые файлы из FOLDER_PATH_DUDL
        try:
            if os.path.exists(FOLDER_PATH_DUDL):
                for filename in os.listdir(FOLDER_PATH_DUDL):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\_Test.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_DUDL, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_DUDL}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_DUDL}': {delete_error}")

        # Удаляем старые файлы из FOLDER_PATH_FEATURES
        try:
            if os.path.exists(FOLDER_PATH_FEATURES):
                for filename in os.listdir(FOLDER_PATH_FEATURES):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\_Test.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_FEATURES, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_FEATURES}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_FEATURES}': {delete_error}")

        # Пытаемся обновить и сохранить файл
        success = update_and_save_excel(file_path_shows_expenses, new_file_path)
        # success = True

        if not success:
            # Если все попытки неудачны, выводим сообщение пользователю
            while not success:
                input("Обновить Excel файл не получилось. Закройте все открытые файлы и нажмите любую кнопку для повторной попытки.")
                success = update_and_save_excel(file_path_shows_expenses, new_file_path)

            print("Файл успешно обновлен после повторной попытки.")

        # После успешного обновления копируем файл в папку FOLDER_PATH_DUDL
        if success:
            try:
                shutil.copy(new_file_path, dudl_file_path)
                print(f"Файл успешно скопирован в папку '{FOLDER_PATH_DUDL}'.")
            except Exception as copy_error:
                print(f"Ошибка при копировании файла в папку '{FOLDER_PATH_DUDL}': {copy_error}")

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Файл успешно обновлен и сохранен. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Показы и затраты ОЗ_2.0.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при обработке файла 'Показы и затраты ОЗ_2.0.xlsx': {e}")

Подготовка данных для ДБ завершена.
Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...
Попытка 1 обновить файл 'Показы и затраты ОЗ_2.0_Test.xlsx'...
Выполняем обновление данных...
